In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Load data
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalise
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten
X_train = X_train.reshape(-1, 32*32*3)
X_test = X_test.reshape(-1, 32*32*3)

# Model builder
def build_model(lr=1e-3):
    model = keras.Sequential()
    
    model.add(layers.Input(shape=(3072,)))
    
    for _ in range(20):
        model.add(
            layers.Dense(
                100,
                activation="elu",
                kernel_initializer="he_normal"
            )
        )
        
    model.add(layers.Dense(10, activation="softmax"))
    
    optimizer = keras.optimizers.Nadam(learning_rate=lr)
    
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )
    
    return model

model = build_model(lr=1e-3) # Most stable learning rate among 1e-2, 3e-3, 1e-3, 3e-4, 1e-4

early_stop = keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 52s 65ms/step - accuracy: 0.2583 - loss: 2.1489 - val_accuracy: 0.3377 - val_loss: 1.8375
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.3480 - loss: 1.7971 - val_accuracy: 0.3483 - val_loss: 1.8204
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.3823 - loss: 1.7082 - val_accuracy: 0.3657 - val_loss: 1.7300
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.4081 - loss: 1.6450 - val_accuracy: 0.3726 - val_loss: 1.7508
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.4268 - loss: 1.5934 - val_accuracy: 0.4154 - val_loss: 1.6426
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.4396 - loss: 1.5607 - val_accuracy: 0.4115 - val_loss: 1.6476
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 46ms/step - accuracy: 0.4552 - loss: 1.5250 - val_accuracy: 0.4286 - val_loss: 1.6246
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 88ms/step - accuracy: 0.4622 - loss: 1.5009 - val

In [5]:
lrs = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4]

results = {}

for lr in lrs:
    print(f"\nTraining with LR = {lr}")
    
    model = build_model(lr)
    
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=5,
        batch_size=128,
        verbose=0
    )
    
    best_val_acc = max(history.history["val_accuracy"])
    results[lr] = best_val_acc
    print("Best val accuracy:", best_val_acc)

print("\nSummary:")
for lr, acc in results.items():
    print(lr, acc)



Training with LR = 0.01
Best val accuracy: 0.3325999975204468

Training with LR = 0.003
Best val accuracy: 0.37560001015663147

Training with LR = 0.001
Best val accuracy: 0.4221000075340271

Training with LR = 0.0003
Best val accuracy: 0.43689998984336853

Training with LR = 0.0001
Best val accuracy: 0.4332999885082245

Summary:
0.01 0.3325999975204468
0.003 0.37560001015663147
0.001 0.4221000075340271
0.0003 0.43689998984336853
0.0001 0.4332999885082245


In [6]:
# batch normalizatin  
def build_model_bn(lr=1e-3):
    model = keras.Sequential()
    
    model.add(layers.Input(shape=(3072,)))
    
    for _ in range(20):
        model.add(layers.Dense(
            100,
            kernel_initializer="he_normal",
            use_bias=False
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation("elu"))
        
    model.add(layers.Dense(10, activation="softmax"))
    
    optimizer = keras.optimizers.Nadam(learning_rate=lr)
    
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )
    
    return model

model_bn = build_model_bn(lr=1e-3)
model_bn.summary()

early_stop = keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model_bn.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[early_stop]
)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_168 (Dense)               │ (None, 100)            │       307,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_169 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_170 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_171 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_172 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_173 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_174 (Dense)               │ (None, 100)            │        10,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 506,210 (1.93 MB)

 Trainable params: 502,210 (1.92 MB)

 Non-trainable params: 4,000 (15.62 KB)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 39ms/step - accuracy: 0.3679 - loss: 1.7750 - val_accuracy: 0.3549 - val_loss: 2.0058
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.4439 - loss: 1.5644 - val_accuracy: 0.3831 - val_loss: 1.7732
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.4763 - loss: 1.4740 - val_accuracy: 0.4203 - val_loss: 1.6640
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.4997 - loss: 1.4095 - val_accuracy: 0.4086 - val_loss: 1.6680
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.5163 - loss: 1.3576 - val_accuracy: 0.4436 - val_loss: 1.6015
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.5337 - loss: 1.3107 - val_accuracy: 0.4502 - val_loss: 1.5690
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.5473 - loss: 1.2734 - val_accuracy: 0.4558 - val_loss: 1.5603
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.5609 - loss: 1.2360 - val

In [7]:
optimizers = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.01),

    "Momentum": tf.keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9
    ),

    "Nesterov": tf.keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9,
        nesterov=True
    ),

    "AdaGrad": tf.keras.optimizers.Adagrad(
        learning_rate=0.01
    ),

    "RMSProp": tf.keras.optimizers.RMSprop(
        learning_rate=0.001
    ),

    "Adam": tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    "Nadam": tf.keras.optimizers.Nadam(
        learning_rate=0.001
    )
}


In [8]:
results = {}

for name, opt in optimizers.items():
    model = build_model()
    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=128,
        verbose=0
    )

    results[name] = history.history


bservations from Task 3:_

higher learning rates can lead to faster convergence but may cause instability.
lower learning rates provide more stable training but  requires more epochs to converge.

Depth alone does not guarantee better performance.

Batch Normalisation stabilises gradient propagation.

Adaptive optimisers significantly accelerate convergence in deep architectures.

For image tasks like CIFAR10, fully connected networks are structurally inferior to CNNs.

By default, the SELU hyperparameters (scale and alpha) are tuned in such a way that the mean output of each neuron remains close to 0, and the standard deviation remains close to 1 (assuming the inputs are standardized with mean 0 and standard deviation 1 too). Using this activation function, even a 1,000 layer deep neural network preserves roughly mean 0 and standard deviation 1 across all layers, avoiding the exploding/vanishing gradients problem: